In [1]:
pip install catboost scikit-learn pandas numpy


Note: you may need to restart the kernel to use updated packages.


In [6]:
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np
import json
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, mean_absolute_error, mean_squared_error
import math

**Парсинг**

In [7]:
xml_path = "/kaggle/input/jerome-result/Jerome.Result.xml"

# Парсинг XML
tree = ET.parse(xml_path)
root = tree.getroot()

data = []
sintagma_num = 0  # Номер текущей синтагмы

def is_fonetic(xml_word):
    for letter in xml_word.findall("letter"):
        flag = int(letter.attrib.get("flag", 0))
        if flag & 1 > 0:
            return True
    return False

for sent in root.iter("sentence"):
    elements = list(sent)
    words_in_sentence = []
    fonetic_count = 0

    for idx, elem in enumerate(elements):
        if elem.tag == "word":
            w_attr = elem.attrib
            content = w_attr.get("original", "")
            stress = w_attr.get("nucleus") == "2"

            di = elem.find("dictitem")
            subpart = di.attrib.get("subpart_of_speech") if di is not None else None
            form = di.attrib.get("form") if di is not None else None
            genesys = di.attrib.get("genesys") if di is not None else None
            sem1 = di.attrib.get("semantics1") if di is not None else None
            sem2 = di.attrib.get("semantics2") if di is not None else None
            stress_dict = di.attrib.get("stress_dict") if di is not None else None

            punkt_beg = punkt_end = emph_beg = emph_end = 0
            if idx > 0 and elements[idx - 1].tag == "content":
                prev_c = elements[idx - 1].attrib
                punkt_beg = int(prev_c.get("PunktBeg", 0))
                emph_beg = int(prev_c.get("EmphBeg", 0))
            if idx + 1 < len(elements) and elements[idx + 1].tag == "content":
                next_c = elements[idx + 1].attrib
                punkt_end = int(next_c.get("PunktEnd", 0))
                emph_end = int(next_c.get("EmphEnd", 0))

            # Паузы
            pause_len = -1
            pause_type = "none"
            has_pause = False
            for j in range(idx + 1, len(elements)):
                if elements[j].tag == "pause":
                    pause_len = int(elements[j].attrib.get("time", -1))
                    pause_type = elements[j].attrib.get("type", "none")
                    has_pause = True
                    break
                if elements[j].tag == "word":
                    break

            # Фонетические слова 
            is_fon = is_fonetic(elem)
            fonetic_before = fonetic_count
            fonetic_count += int(is_fon)

            # Контекст 
            num_words = sum(1 for e in elements if e.tag == "word")
            order = len(words_in_sentence) + 1
            words_before = len(words_in_sentence)
            words_after = num_words - order

            prev_word = words_in_sentence[-1]["content"] if words_in_sentence else ""
            next_word = ""
            # Определим следующее слово позже при построении синтагмы

            is_capital = int(content[:1].isupper())
            word_len = len(content)

            word_info = {
                "content": content,
                "phrasal_stress": stress,
                "pause_len": pause_len,
                "pause_type": pause_type,
                "has_pause": has_pause,
                "subpart": subpart,
                "form": form,
                "genesys": genesys,
                "sem1": sem1,
                "sem2": sem2,
                "stress_dict": stress_dict,
                "punkt_beg": punkt_beg,
                "punkt_end": punkt_end,
                "emph_beg": emph_beg,
                "emph_end": emph_end,
                "word_len": word_len,
                "is_capital": is_capital,
                "order": order,
                "words_before": words_before,
                "words_after": words_after,
                "prev_word": prev_word,
                "next_word": next_word,  # заполним позже
                "fonetic_words_before": fonetic_before,
                "sintagma_num": sintagma_num,
            }

            words_in_sentence.append(word_info)

        elif elem.tag == "pause":
            # Конец синтагмы - обновляем номер
            sintagma_num += 1
            if words_in_sentence:
                words_in_sentence[-1]["has_pause"] = True
                words_in_sentence[-1]["pause_len"] = int(elem.attrib.get("time", -1))
                words_in_sentence[-1]["pause_type"] = elem.attrib.get("type", "none")
                words_in_sentence[-1]["sintagma_num"] = sintagma_num

    for i, w in enumerate(words_in_sentence):
        w["next_word"] = words_in_sentence[i + 1]["content"] if i + 1 < len(words_in_sentence) else ""
        w["fonetic_words_after"] = fonetic_count - w["fonetic_words_before"] - 1
        data.append(w)

df = pd.DataFrame(data)
print(f"Всего слов: {len(df)}")
print(f"Количество синтагм: {df['sintagma_num'].nunique()}")

Всего слов: 42441
Количество синтагм: 8413


**Подготовка данных**

In [8]:
cat_cols = [
    "subpart", "form", "genesys", "sem1", "sem2",
    "stress_dict", "pause_type", "prev_word", "next_word"
]
num_cols = [
    "word_len", "is_capital", "punkt_beg", "punkt_end",
    "emph_beg", "emph_end", "order", "words_before", "words_after",
    "fonetic_words_before", "fonetic_words_after"
]

for c in cat_cols:
    df[c] = df[c].fillna("unknown").astype(str)

X = df[cat_cols + num_cols]
y_cls = df["phrasal_stress"].astype(int)
y_reg = df["pause_len"].astype(int)


X_train, X_test, y_cls_train, y_cls_test, y_reg_train, y_reg_test, df_train_idx, df_test_idx = train_test_split(
    X, y_cls, y_reg, df.index, test_size=0.2, random_state=42, stratify=y_cls
)

**Обучение классификатора и регрессора**

In [9]:
#  Обучение моделей 
cls_model = CatBoostClassifier(
    iterations=300,
    depth=8,
    learning_rate=0.1,
    loss_function="Logloss",
    cat_features=cat_cols,
    verbose=False,
    random_seed=42
)

reg_model = CatBoostRegressor(
    iterations=300,
    depth=8,
    learning_rate=0.1,
    loss_function="RMSE",
    cat_features=cat_cols,
    verbose=False,
    random_seed=42
)

print("Обучаем классификатор (фразовое ударение)...")
cls_model.fit(X_train, y_cls_train)
print("Обучаем регрессор (длина паузы)...")
reg_model.fit(X_train, y_reg_train)

Обучаем классификатор (фразовое ударение)...
Обучаем регрессор (длина паузы)...


**Оценка и сохранение в json**

In [15]:
y_pred_cls = cls_model.predict(X_test)
y_pred_reg = reg_model.predict(X_test)

print("\n Классификатор")
print(classification_report(y_cls_test, y_pred_cls, digits=3))

print("\n Регрессор")
mae = mean_absolute_error(y_reg_test, y_pred_reg)
rmse = math.sqrt(mean_squared_error(y_reg_test, y_pred_reg))
print(f"MAE  (pause length): {mae:.3f}")
print(f"RMSE (pause length): {rmse:.3f}")


json_result = [{
    "words": [
        {
            "content": df.loc[idx, "content"],
            "phrasal_stress": bool(cls_pred),
            "pause_len": int(round(reg_pred)) if reg_pred > 0 else -1
        }
        for idx, cls_pred, reg_pred in zip(df_test_idx, y_pred_cls, y_pred_reg)
    ]
}]

with open("result.json", "w", encoding="utf-8") as f:
    json.dump(json_result, f, ensure_ascii=False, indent=4)

print("\n Сохранено в .json")


 Классификатор
              precision    recall  f1-score   support

           0      0.990     0.993     0.992      6807
           1      0.972     0.960     0.966      1682

    accuracy                          0.987      8489
   macro avg      0.981     0.977     0.979      8489
weighted avg      0.987     0.987     0.987      8489


 Регрессор
MAE  (pause length): 16.242
RMSE (pause length): 43.779

 Сохранено в .json


In [12]:
print("\n важность признаков (Classifier)")
for name, val in zip(X.columns, cls_model.get_feature_importance()):
    print(f"{name:20s} {val:.3f}")

print("\n Важность признаков (Regressor)")
for name, val in zip(X.columns, reg_model.get_feature_importance()):
    print(f"{name:20s} {val:.3f}")


 важность признаков (Classifier)
subpart              4.526
form                 4.103
genesys              1.591
sem1                 1.489
sem2                 4.071
stress_dict          9.734
pause_type           41.388
prev_word            5.121
next_word            6.232
word_len             4.401
is_capital           0.092
punkt_beg            0.838
punkt_end            4.848
emph_beg             0.046
emph_end             0.091
order                0.889
words_before         1.080
words_after          2.644
fonetic_words_before 1.749
fonetic_words_after  5.069

 Важность признаков (Regressor)
subpart              0.414
form                 0.975
genesys              0.804
sem1                 0.293
sem2                 0.682
stress_dict          0.660
pause_type           81.153
prev_word            0.641
next_word            1.173
word_len             0.590
is_capital           0.054
punkt_beg            0.012
punkt_end            0.000
emph_beg             0.039
emph_end     